# Trajectory Pilot — Diagnostic Notebook

验证 `run_trajectory_pilot.ipynb` 产出的 3 个 run（V / V+T / V+K+T）为什么 val_ccc ≈ 0、val_mse 从 epoch 2 起无法下降。

按顺序跑 3 个实验：

| # | 实验 | 耗时 | 目的 |
|---|---|---|---|
| 1 | 零预测 baseline | ~3 min | 验证 `ckpt_best.pt` (epoch 2) 的 MSE 是否仅等于 "predict 0" |
| 2 | eval `ckpt_last.pt` (epoch 10) | ~3 min | 训练后模型是否学到了东西（看 pred_std 和 overfitting 方向） |
| 3 | Uniform sampling 重跑 single_video | ~10 min | 关掉 event-weighted sampler，看 val_mse 能否下降过 epoch 2 |

**决策规则**：
- 实验 1 发现 `best ≈ zero` → 确认根因 A+B（ckpt_best ≈ 未训练）
- 实验 2 发现 `last >> zero` → 确认根因 B（训练加剧 val 偏差 = 过拟合）
- 实验 3 发现 uniform run 的 val_mse 能持续下降且 < event-weighted → 确认根因 C（采样不对称）

**前置条件**：已跑过 `run_trajectory_pilot.ipynb` 和 `run_trajectory_eval.ipynb`，产生 `metrics.json` + `stratified_eval.json`。

In [1]:
# Cell 1: Mount + cd
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ProjectExperiment
NVIDIA L4, 23034 MiB


In [2]:
# Cell 2: Locate the 3 existing pilot runs (same logic as run_trajectory_eval.ipynb)
import glob, json
from pathlib import Path

BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed'

run_dirs = {}
for mod_tag in ['single_video', 'dual_video_telem', 'triple_video_km_telem']:
    matches = sorted(glob.glob(f'{BASE}/{mod_tag}/*seed0*'))
    valid = [m for m in matches if (Path(m) / 'ckpt_best.pt').exists()]
    assert valid, f'no completed run dir with ckpt_best.pt for {mod_tag}; candidates: {matches}'
    run_dirs[mod_tag] = valid[-1]
    print(f'{mod_tag}: {run_dirs[mod_tag]}')

single_video: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0
dual_video_telem: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/dual_video_telem/2026-04-23_19-35-24__amucs_trajectory__eft__telem_video__seed0
triple_video_km_telem: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/triple_video_km_telem/2026-04-23_19-43-54__amucs_trajectory__eft__km_telem_video__seed0


---
## Exp 1 — 零预测 baseline

构造与训练相同的归一化 datamodule，iterate val/test loader，计算 `(y ** 2).mean()`。

如果 `ckpt_best.pt` 的 overall MSE ≈ 这个值，就是 predict-zero 的水平。

In [3]:
# Cell 3: Compute zero-prediction MSE on val/test per modality
import sys, yaml
if '/content/drive/MyDrive/ProjectExperiment' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/ProjectExperiment')

import torch
import src.data.datamodules  # triggers @DATAMODULES.register
from src.core.registry import DATAMODULES

zero_mse_results = {}

for mod_tag, run_dir in run_dirs.items():
    print(f'\n--- Building datamodule for {mod_tag} ---', flush=True)
    cfg = yaml.safe_load(open(Path(run_dir) / 'config.yaml'))
    data_cfg = dict(cfg['data'])
    data_cfg['batch_size'] = 256
    data_cfg['num_workers'] = 4
    data_cfg['event_weighted'] = False   # skip weight computation — we only read val/test

    dm = DATAMODULES.build('amucs_trajectory', data_cfg)

    def collect_sq(loader):
        if loader is None:
            return float('nan')
        total_sq, n = 0.0, 0
        for batch in loader:
            y = batch['y'].float()
            total_sq += (y ** 2).sum().item()
            n += y.numel()
        return total_sq / max(n, 1)

    val_mse = collect_sq(dm.val_dataloader())
    test_mse = collect_sq(dm.test_dataloader())
    zero_mse_results[mod_tag] = {
        'val': val_mse, 'test': test_mse, 'delta_sigma': dm.delta_sigma,
    }
    print(f'{mod_tag:25} Δσ_train={dm.delta_sigma:.3f}  zero_val_mse={val_mse:.4f}  zero_test_mse={test_mse:.4f}')

print('\n' + '=' * 80)
print('对照 ckpt_best.pt (epoch 2) 的实测 MSE（来自 stratified_eval.json overall）:')
print('=' * 80)
print(f"{'modality':<25} {'split':<6} {'zero_mse':>10} {'best_mse':>10} {'ratio':>8}  {'verdict':<30}")
print('-' * 80)
for mod_tag, run_dir in run_dirs.items():
    se_path = Path(run_dir) / 'stratified_eval.json'
    if not se_path.exists():
        print(f'{mod_tag}: no stratified_eval.json — run run_trajectory_eval.ipynb first')
        continue
    se = json.load(open(se_path))
    z = zero_mse_results[mod_tag]
    for split in ['val', 'test']:
        best = se['results'][split]['overall']['mse']
        zero = z[split]
        ratio = best / zero if zero > 0 else float('nan')
        verdict = 'BEATS zero' if best < zero * 0.95 else ('≈ zero (no learning)' if best < zero * 1.05 else 'WORSE than zero')
        print(f'{mod_tag:<25} {split:<6} {zero:>10.4f} {best:>10.4f} {ratio:>8.3f}  {verdict:<30}')


--- Building datamodule for single_video ---
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4468
[AMuCSTrajectoryDataModule] train sampler = uniform (event_weighted=false)
single_video              Δσ_train=18.447  zero_val_mse=1.4980  zero_test_mse=2.4473

--- Building datamodule for dual_video_telem ---
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4869
[AMuCSTrajectoryDataModule] train sampler = uniform (event_weighted=false)
dual_video_telem          Δσ_train=18.487  zero_val_mse=1.4692  zero_test_mse=2.2453

--- Building datamodule for triple_video_km_telem ---
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4869
[AMuCSTrajectoryDataModule] train sampler = uniform (event_weighted=false)
triple_video_km_telem     Δσ_train=18.487  zero_val_mse=1.4692  zero_test_mse=2.2453

对照 ckpt_best.pt (epoch 2) 的实测 MSE（来自 stratified_eval.json overall）:
modality                  split    zero_mse   best_mse    ratio  verdict                       
---------------------------------------------------

---
## Exp 2 — eval `ckpt_last.pt` (epoch 10)

训练日志显示 `ckpt_best.pt` 保存在 epoch 2。后面 8 个 epoch 里 val_mse 持续变差 —— 如果真是过拟合，epoch 10 的模型应该 `pred_std` 更大且 `val_mse >> zero_baseline`。

用修改后的 `evaluate_trajectory.py --ckpt_name ckpt_last.pt` 跑 3 个 run。结果写到 `stratified_eval_ckpt_last.json`。

In [4]:
# Cell 4: Run stratified eval on ckpt_last.pt for each modality
import subprocess, sys

for mod_tag, run_dir in run_dirs.items():
    out_path = Path(run_dir) / 'stratified_eval_ckpt_last.json'
    if out_path.exists():
        print(f'[SKIP] {mod_tag} — {out_path.name} already exists')
        continue
    print(f'\n{"="*70}\nEvaluating {mod_tag} / ckpt_last.pt\n{"="*70}', flush=True)
    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/evaluate_trajectory.py',
         '--run_dir', run_dir, '--splits', 'val', 'test',
         '--ckpt_name', 'ckpt_last.pt'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'!!! {mod_tag} exited with code {proc.returncode}')


Evaluating single_video / ckpt_last.pt
[load] config from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0/config.yaml
[load] checkpoint from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_3seed/single_video/2026-04-23_19-30-36__amucs_trajectory__eft__video__seed0/ckpt_last.pt
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4468
[AMuCSTrajectoryDataModule] train sampler = event-weighted (tau_s=10.0)
[val] collecting predictions ...
[val] 6259 anchors  pred_shape=(6259, 51)  target_shape=(6259, 51)
[val] strata: event=4999 (79.9%) quiet=1260 (20.1%)
[test] collecting predictions ...
[test] 13493 anchors  pred_shape=(13493, 51)  target_shape=(13493, 51)
[test] strata: event=9966 (73.9%) quiet=3527 (26.1%)

=== VAL ===
stratum        n      mse     ccc pred_std  gt_std  peak_f1  lead_s  amp_err  ev_corr
---------------

In [5]:
# Cell 5: Summary — zero | best(ep2) | last(ep10) 三方对比
def fmt(v, w=10, d=4):
    if v is None or (isinstance(v, float) and v != v):
        return f'{"nan":>{w}}'
    return f'{v:>{w}.{d}f}'

print(f"{'modality':<25} {'split':<6} | {'zero':>8} | {'best(ep2)':>10} | {'last(ep10)':>11} | {'ps_best':>8} {'ps_last':>8} | verdict")
print('-' * 120)
for mod_tag, run_dir in run_dirs.items():
    z = zero_mse_results[mod_tag]
    se_best = json.load(open(Path(run_dir) / 'stratified_eval.json'))
    se_last_path = Path(run_dir) / 'stratified_eval_ckpt_last.json'
    if not se_last_path.exists():
        print(f'{mod_tag}: missing stratified_eval_ckpt_last.json — run Cell 4 first')
        continue
    se_last = json.load(open(se_last_path))
    for split in ['val', 'test']:
        zm = z[split]
        b_ov = se_best['results'][split]['overall']
        l_ov = se_last['results'][split]['overall']
        b_mse, b_ps = b_ov['mse'], b_ov.get('pred_std', float('nan'))
        l_mse, l_ps = l_ov['mse'], l_ov.get('pred_std', float('nan'))
        if l_mse > zm * 1.15:
            verdict = 'OVERFIT (last >> zero)'
        elif l_mse < zm * 0.95:
            verdict = 'last BEATS zero'
        elif abs(b_mse - zm) / zm < 0.05 and l_ps < 0.15:
            verdict = 'best ≈ zero (no learning)'
        else:
            verdict = 'ambiguous'
        print(f'{mod_tag:<25} {split:<6} | {fmt(zm, 8)} | {fmt(b_mse, 10)} | {fmt(l_mse, 11)} | {fmt(b_ps, 8, 3)} {fmt(l_ps, 8, 3)} | {verdict}')

print()
print('解读:')
print('  - best ≈ zero    → ckpt_best.pt = 近似未训练，eval notebook 看到的 "constant" 就是这个')
print('  - last >> zero   → 训练确实在进行，但学的方向不 generalize 到 val (→ 根因 C 或 D)')
print('  - ps_last > ps_best → 训练使输出 std 变大，证明模型学到了 something（只是方向错）')

modality                  split  |     zero |  best(ep2) |  last(ep10) |  ps_best  ps_last | verdict
------------------------------------------------------------------------------------------------------------------------
single_video              val    |   1.4980 |     1.4981 |      1.6362 |    0.062    0.379 | ambiguous
single_video              test   |   2.4473 |     2.4544 |      2.7503 |    0.063    0.528 | ambiguous
dual_video_telem          val    |   1.4692 |     1.4686 |      1.6093 |    0.066    0.371 | ambiguous
dual_video_telem          test   |   2.2453 |     2.2518 |      2.3727 |    0.069    0.330 | ambiguous
triple_video_km_telem     val    |   1.4692 |     1.4670 |      1.5368 |    0.059    0.311 | ambiguous
triple_video_km_telem     test   |   2.2453 |     2.2497 |      2.2900 |    0.065    0.215 | ambiguous

解读:
  - best ≈ zero    → ckpt_best.pt = 近似未训练，eval notebook 看到的 "constant" 就是这个
  - last >> zero   → 训练确实在进行，但学的方向不 generalize 到 val (→ 根因 C 或 D)
  - ps_last >

---
## Exp 3 — Uniform sampling 重跑 single_video

`configs/sweeps/trajectory_pilot_uniform.yaml` 和 pilot 的唯一差异是 `shared.data.event_weighted: false` —— 关掉事件加权，train 改用均匀采样，和 val/test 分布对齐。

只跑 1 个 run（single_video / seed0），~10 min。

**预期**：
- 如果 val_mse best_epoch > 2 且 val_mse 持续下降 → 确认根因 C（采样不对称是主因）
- 如果仍然 best_epoch=2 → 说明根因 D（cross-subject Δσ 漂移）或目标本身不可预测才是主因

In [6]:
# Cell 6: Run the uniform sampling sweep (1 run, ~10 min)
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot_uniform.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot

Sweep plan: 1 runs total, 0 already completed, 1 to run


[1/1] [cross_subject] eft_trajectory_pilot_uniform / single_video / seed0
[AMuCSTrajectoryDataModule] train-set Δσ = 18.4468
[AMuCSTrajectoryDataModule] train sampler = uniform (event_weighted=false)
torch.compile enabled
Model parameters: 8,950,323 (trainable: 8,950,323)
Train samples: 19123, Val samples: 6259
Run directory: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_uniform_3seed/single_video/2026-04-24_09-57-46__amucs_trajectory__eft__video__seed0
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0424 09:58:05.720000 21392 torch/_inductor/utils.py:1679] [9/0] Not enough SMs to use max_autotune_gemm mode
Epoch 1/40 | train_ccc: 0.0011 | train_loss:

In [7]:
# Cell 7: Compare uniform run to original event-weighted single_video run
UNIFORM_BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_uniform_3seed/single_video'
uniform_candidates = sorted(glob.glob(f'{UNIFORM_BASE}/*seed0*'))
uniform_valid = [m for m in uniform_candidates if (Path(m) / 'metrics.json').exists()]
assert uniform_valid, f'no completed uniform run under {UNIFORM_BASE}'
uniform_run = uniform_valid[-1]
print(f'uniform run: {uniform_run}\n')

def read_summary(run_dir, label):
    m = json.load(open(Path(run_dir) / 'metrics.json'))
    return {
        'label': label,
        'best_epoch': m.get('best_epoch'),
        'best_val_mse': m.get('best_val_metric'),
        'total_epochs': m.get('total_epochs'),
        'early_stopped': m.get('early_stopped'),
        'test_mse': m.get('test_mse'),
        'test_ccc': m.get('test_ccc'),
    }

summaries = [
    read_summary(run_dirs['single_video'], 'event-weighted (original)'),
    read_summary(uniform_run, 'uniform (new)'),
]

zero_val = zero_mse_results['single_video']['val']
zero_test = zero_mse_results['single_video']['test']

print(f"{'run':<28} {'best_ep':>8} {'best_val_mse':>13} {'total_ep':>9} {'test_mse':>9} {'test_ccc':>9}")
print('-' * 90)
for s in summaries:
    ep = s['best_epoch']
    bvm = s['best_val_mse']
    te = s['total_epochs']
    tm = s['test_mse']
    tc = s['test_ccc']
    print(f"{s['label']:<28} {ep!s:>8} {(bvm if isinstance(bvm,float) else '?')!s:>13} {te!s:>9} {(tm if isinstance(tm,float) else '?')!s:>9} {(tc if isinstance(tc,float) else '?')!s:>9}")

print(f'\nzero-prediction baselines: val={zero_val:.4f}  test={zero_test:.4f}')

orig, uni = summaries
if uni['best_epoch'] is not None and orig['best_epoch'] is not None:
    print()
    if uni['best_epoch'] > orig['best_epoch'] + 2 and uni['best_val_mse'] < orig['best_val_mse']:
        print('✅ 根因 C 成立：uniform sampling 让训练能持续改善 val_mse')
    elif uni['best_epoch'] <= 3 and uni['best_val_mse'] >= zero_val * 0.95:
        print('❌ 根因 C 不足以解释问题：uniform 仍然 epoch<=3 且未打过 zero baseline')
        print('   → 需要进一步检查根因 D（Δσ 漂移）或目标本身可预测性')
    else:
        print('🟡 部分改善：uniform 有效但非全部根因')

uniform run: /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/cross_subject/eft_trajectory_pilot_uniform_3seed/single_video/2026-04-24_09-57-46__amucs_trajectory__eft__video__seed0

run                           best_ep  best_val_mse  total_ep  test_mse  test_ccc
------------------------------------------------------------------------------------------
event-weighted (original)           2 1.5100682973861694        10 2.7387807369232178 -0.009347598999738693
uniform (new)                       2 1.5089508295059204        10 2.64597487449646 0.002120871562510729

zero-prediction baselines: val=1.4980  test=2.4473

❌ 根因 C 不足以解释问题：uniform 仍然 epoch<=3 且未打过 zero baseline
   → 需要进一步检查根因 D（Δσ 漂移）或目标本身可预测性


---
## 结论判读

跑完后回到 Cell 3、Cell 5、Cell 7 的打印，按以下规则读：

### Cell 3（零预测 baseline）
- `best_mse / zero_mse ∈ [0.95, 1.05]` → `ckpt_best.pt` 就是零预测水平
- `best_mse < zero_mse * 0.95` → 模型其实学到了一点

### Cell 5（ckpt_last vs ckpt_best）
- `last_mse >> best_mse` → epoch 2 后训练加剧过拟合（符合训练日志）
- `ps_last > ps_best × 2` → 训练让输出 std 增长，证明梯度确实在更新

### Cell 7（uniform vs event-weighted）
- `uniform.best_epoch > 5` 且 `uniform.best_val_mse < event_weighted.best_val_mse` → 根因 C 是主要原因
- `uniform.best_epoch ≤ 3` 且 `uniform.best_val_mse ≈ zero` → 采样不是主因，根因可能是 D 或任务本身

### 下一步取决于结果

| Cell 3 | Cell 5 | Cell 7 | 推荐下一步 |
|---|---|---|---|
| best ≈ zero | last >> zero | uniform 改善 | 正式 sweep 3 模态 × 3 seed，换 uniform 配置 |
| best ≈ zero | last >> zero | uniform 不改善 | 检查 Δσ 跨 subject 漂移，考虑 per-subject 归一化 |
| best ≈ zero | last ≈ best | uniform 不改善 | 模型根本没学（检查学习率、梯度、数据是否进模型） |
| best < zero | 任意 | 任意 | 我上轮的诊断错了，重新排查 |